## Silver層 Bronze層の生データに対するクレンジング・型変換・重複排除

### 概要
Bronze層で取り込んだ生データに対して、クレンジング・型変換・重複排除を行い、分析や集計に適した品質の高いデータに整形する。

### 処理フロー
**1. データクレンジング&型変換**
- 不要なID列(インデックス列)の除外
- 文字列や数値の型キャスト・正規化
- 異常値(サイズが0以下のレコードなど)のフィルタリング

**2. Merge Into(Upsert)による冪等性の確保**
- 単純なAppend(追記)ではなく、主キー(または複合キー)をベースに、データが存在する場合は更新し無ければ追加する処理を同時に行う。

In [0]:
from pyspark.sql.functions import col, expr, current_timestamp
from delta.tables import DeltaTable

# 1. カタログとスキーマ設定
spark.sql("USE CATALOG my_ecommerce_pipeline")
spark.sql("USE SCHEMA silver")

# 2. Bronzeデータの読み込み(バッチ読み込み)
bronze_df = spark.read.table("bronze.raw_diamonds")

# 3. データクレンジング
# NULL除去、0未満のデータを削除、必要カラムのみ抽出、重複排除
cleaned_df = (
    bronze_df
    .filter( col("carat").isNotNull() & (col("price") > 0) )
    .filter( (col("x") > 0.0) & (col("y") > 0.0) & (col("z") > 0.0) )
    .select(
        col("carat").cast("double").alias("carat"),
        col("cut").cast("string").alias("cut"),
        col("color").cast("string").alias("color"),
        col("clarity").cast("string").alias("clarity"),
        col("depth").cast("double").alias("depth"),
        col("table").cast("double").alias("table_pct"),
        col("price").cast("int").alias("price"),
        col("x").cast("double").alias("length_x"),
        col("y").cast("double").alias("width_y"),
        col("z").cast("double").alias("depth_z"),
        col("_ingested_at"),
        current_timestamp().alias("_updated_at")
    )
    # 重複排除(1レコードに集約)
    .dropDuplicates(["carat", "cut", "color", "clarity", "depth", "table_pct", "price", "length_x", "width_y", "depth_z"])
)

# 4. Target(Silverテーブル)の事前作成(初回実行時のみテーブルを初期化)
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.cleaned_diamonds (
    carat DOUBLE,
    cut STRING,
    color STRING,
    clarity STRING,
    depth DOUBLE,
    table_pct DOUBLE,
    price INT,
    length_x DOUBLE,
    width_y DOUBLE,
    depth_z DOUBLE,
    _ingested_at TIMESTAMP,
    _updated_at TIMESTAMP
) USING DELTA
""")

# 5. Delta MERGE(Upsert)処理によるデータの反映(存在する場合は更新、無ければ追加する処理を同時に行う)
target_table = DeltaTable.forName(spark, "silver.cleaned_diamonds")

# 複合キーによるバッチMERGE条件の設定
# 全ての値が一致したら同じデータとみなす
merge_condition = """
    target.carat = source.carat AND
    target.cut = source.cut AND
    target.color = source.color AND
    target.clarity = source.clarity AND
    target.depth = source.depth AND
    target.table_pct = source.table_pct AND
    target.price = source.price AND
    target.length_x = source.length_x AND
    target.width_y = source.width_y AND
    target.depth_z = source.depth_z
"""

(
    target_table.alias("target")
    .merge(cleaned_df.alias("source"), merge_condition) # クレンジングしたデータをmerge_conditionと照合
    .whenMatchedUpdateAll()      # 条件に合致したら更新
    .whenNotMatchedInsertAll()   # 条件に合致しなかったら追加
    .execute()
)

print("Silver層への変換・MERGE処理が完了しました!")